In [42]:
# %pip install PyYAML
# %pip install dynamic-yaml


In [ ]:
import dynamic_yaml as yaml
import os
os.environ["DATA_FOLDER_PATH"] = r"/home/user/data-da-ds-de/data_validation_pandas/data_test/raw/parquet"
# import json
# import pandas as pd

# with open("config.yml", "r") as f:
#     data = yaml.load(f)
#     df = pd.DataFrame(data)

# df

In [4]:
import yaml
import json
import pandas as pd

with open(r"C:\Users\ASUS\Code\z_project\data_validation\configs\validation_config.yml", "r") as f:
    data = yaml.load(f, Loader=yaml.FullLoader)
    df = pd.DataFrame(data)
    print(type(data), json.dumps(data, indent=2))

<class 'dict'> {
  "files": [
    {
      "sheet_name": "InternalSuspendedList",
      "file_path": "${DATA_FOLDER_PATH}/InternalSuspendedList.parquet",
      "file_type": "parquet",
      "report_file_path": "${DATA_FOLDER_PATH}/InternalSuspendedList_report.xlsx",
      "columns": [
        {
          "name": "ID",
          "data_type": "string",
          "required": true,
          "skip_if_null": false,
          "rules": [
            {
              "type": "mandatory"
            },
            {
              "type": "unique"
            }
          ]
        },
        {
          "name": "Debtor type",
          "data_type": "string",
          "required": true,
          "skip_if_null": false,
          "rules": [
            {
              "type": "value_list",
              "values": [
                "Individual",
                "Company"
              ]
            }
          ]
        },
        {
          "name": "NRIC/FIN",
          "data_type": "string",
     

In [45]:
df_files = pd.json_normalize(df["files"]).explode("columns", ignore_index=True)
df_columns = pd.json_normalize(df_files["columns"]).explode("rules", ignore_index=True)
df_files = pd.concat([df_files,df_columns], axis=1)
df_rules = pd.json_normalize(df_files["rules"])
df_files = pd.concat([df_files,df_rules], axis=1)

df_files

,sheet_name,file_path,file_type,report_file_path,columns,name,data_type,required,is_pre_processing,skip_if_null,rules,type,values,ref_file,ref_column,ref_values,format,min,max,decimals
0,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'ID', 'data_type': 'string', 'require...",ID,string,True,False,False,{'type': 'not_null'},not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'Debtor type', 'data_type': 'string',...",ID,string,True,False,False,{'type': 'unique'},unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'NRIC/FIN', 'data_type': 'string', 'r...",Debtor type,string,True,False,False,"{'type': 'value_list', 'values': ['Individual'...",value_list,"[Individual, Company]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'Suspended reason', 'data_type': 'str...",NRIC/FIN,string,True,False,False,{'type': 'not_null'},not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'Effective from', 'data_type': 'date'...",NRIC/FIN,string,True,False,False,{'type': 'unique'},unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'Bad debt write-off amount', 'data_ty...",Suspended reason,string,True,True,False,{'type': 'not_null'},not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,InternalSuspendedList,${DATA_FOLDER_PATH}/InternalSuspendedList.parquet,parquet,${DATA_FOLDER_PATH}/InternalSuspendedList_repo...,"{'name': 'Status', 'data_type': 'string', 'req...",Suspended reason,string,True,True,False,"{'type': 'reference', 'ref_file': '${DATA_FOLD...",reference,NaN,${DATA_FOLDER_PATH}/SuspensionReason.parquet,suspension_reason_name,All,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,Effective from,date,True,False,False,{'type': 'not_null'},not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,Effective from,date,True,False,False,"{'type': 'date_format', 'format': '%d/%m/%Y'}",date_format,NaN,NaN,NaN,NaN,%d/%m/%Y,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,Effective from,date,True,False,False,"{'type': 'date_range', 'min': '01/01/1900', 'm...",date_range,NaN,NaN,NaN,NaN,NaN,01/01/1900,31/12/2099,NaN


In [46]:
df_files["file_path"] = df_files["file_path"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])
df_files.loc[0, "file_path"]
df_files["report_file_path"] = df_files["report_file_path"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])
df_files["ref_file"] = df_files["ref_file"].str.replace("${DATA_FOLDER_PATH}", os.environ["DATA_FOLDER_PATH"])

In [47]:
df_files_copy = df_files.copy()
df_files_copy.drop(["columns", "rules"], axis=1, inplace=True)
df_files_copy[[
    "sheet_name",
    "file_path",
    "file_type",
    "report_file_path"
]] = df_files_copy[[
    "sheet_name",
    "file_path",
    "file_type",
    "report_file_path"
]].ffill()
df_files_copy

,sheet_name,file_path,file_type,report_file_path,name,data_type,required,is_pre_processing,skip_if_null,type,values,ref_file,ref_column,ref_values,format,min,max,decimals
0,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,ID,string,True,False,False,not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,ID,string,True,False,False,unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,Debtor type,string,True,False,False,value_list,"[Individual, Company]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,NRIC/FIN,string,True,False,False,not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,NRIC/FIN,string,True,False,False,unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,Suspended reason,string,True,True,False,not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,Suspended reason,string,True,True,False,reference,NaN,/home/user/data-da-ds-de/data_validation_panda...,suspension_reason_name,All,NaN,NaN,NaN,NaN
7,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,Effective from,date,True,False,False,not_null,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,Effective from,date,True,False,False,date_format,NaN,NaN,NaN,NaN,%d/%m/%Y,NaN,NaN,NaN
9,InternalSuspendedList,/home/user/data-da-ds-de/data_validation_panda...,parquet,/home/user/data-da-ds-de/data_validation_panda...,Effective from,date,True,False,False,date_range,NaN,NaN,NaN,NaN,NaN,01/01/1900,31/12/2099,NaN
